# Project - Use a bag of words approach to easily examine papers.

### Intersting ideas to explore:

* Do the embeddings of abstracts closely match the embeddings of the paper? (i.e. Does the abstract accurately summarize the rest of the document?)

* Do similar abstract embeddings correlate to similar papers? Do the closest abstracts appear as references across papers?

### Pull in processed papers

In [33]:
# load in preprocessed papers

import json
from pathlib import Path

def load_processed():
    processed = []
    for file in Path("processed").glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            processed.append(json.load(f))
    return processed

papers = load_processed()
print(papers[0].keys())  # dict with abstract, body, references


dict_keys(['filename', 'abstract', 'body', 'references'])


In [2]:
# Load in pretrained fastText embeddings

import gensim.downloader as api
ft_model = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [3]:
# document embedding function

import numpy as np
import re

def embed_doc(doc, model):
    # Basic tokenization: lowercase, split on non-letters
    tokens = re.findall(r"\b\w+\b", doc.lower())
    
    vectors = [model[w] for w in tokens if w in model]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)


In [34]:
# compute abstract and body embeddings for comparison

abstract_embeddings = []
body_embeddings = []
filenames = []

for paper in papers:
    abs_vec = embed_doc(paper["abstract"], ft_model)
    body_vec = embed_doc(paper["body"], ft_model)
    
    abstract_embeddings.append(abs_vec)
    body_embeddings.append(body_vec)
    filenames.append(paper["filename"])


In [35]:
# save embeddings so we don't have to recompute later

import pickle

with open("processed/fasttext_embeddings.pkl", "wb") as f:
    pickle.dump({
        "abstracts": abstract_embeddings,
        "bodies": body_embeddings,
        "filenames": filenames
    }, f)


In [36]:
# reload those embeddings

with open("processed/fasttext_embeddings.pkl", "rb") as f:
    data = pickle.load(f)

abstract_embeddings = data["abstracts"]
body_embeddings = data["bodies"]
filenames = data["filenames"]


In [14]:
# comparing cosine similarities

import numpy as np

def cosine_similarity(vec1,vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0 # handle empty embeddings
    return np.dot(vec1,vec2) / (norm1 * norm2)

def compare_abstract_paper(index: int) -> float:
    return cosine_similarity(abstract_embeddings[index],body_embeddings[index])

def compare_abstracts(index1: int, index2: int) -> float:
    return cosine_similarity(abstract_embeddings[index1],abstract_embeddings[index2])

def compare_papers(index1: int, index2: int) -> float:
    return cosine_similarity(body_embeddings[index1],body_embeddings[index2])

### First results
The initial tests were run comparing the papers that we were assigned to read in class. When comparing the abstracts, we see that there is a higher similarity amongst abstracts than between a given paper's abstract and its body sections. I made the initial guess that this could be because of the similar structure of abstracts and the generalization, so I implemented a function to compare the bodies of papers and added that. It turns out that the papers are, if anything, more similar than the abstracts across papers. 

The fact that papers on similar, though distinct topics are more similar than the comparison of any of the paper's abstracts are with their own bodies is not good for supporting my hope that abstract embeddings can help identify the body of the paper. 

I choose to then compare the similarity of one abstract to the all papers' body sections, then introduce papers from unrelated fields to further explore. 

In [17]:
abstract_similarity_matrix = []
body_similarity_matrix = []
for i in range(4):
    tmp_abstract = []
    tmp_body = []
    for j in range(4):
        tmp_abstract.append(round(compare_abstracts(i,j),4))
        tmp_body.append(round(compare_papers(i,j),4))
    abstract_similarity_matrix.append(tmp_abstract)
    body_similarity_matrix.append(tmp_body)

print('==== abstract comparisons ====')
for row in abstract_similarity_matrix:
    print(row)

print('\n====== body comparisons ======')
for row in  body_similarity_matrix:
    print(row)

print('\n ====== abstract/body comparisons ======')
for i in range(4):
    print(f'file: {filenames[i]}')
    print(f'\tpaper/abstract similarity: {compare_abstract_paper(i)}\n')

==== abstract comparisons ====
[1.0, 0.9872, 0.9879, 0.9844]
[0.9872, 1.0, 0.9869, 0.9853]
[0.9879, 0.9869, 1.0, 0.9809]
[0.9844, 0.9853, 0.9809, 1.0]

====== body comparisons ======
[1.0, 0.9898, 0.9881, 0.9916]
[0.9898, 1.0, 0.9944, 0.9942]
[0.9881, 0.9944, 1.0, 0.9948]
[0.9916, 0.9942, 0.9948, 1.0]

 ====== abstract/body comparisons ======
file: DistributedRepresentationsofWordsandPhrasesandtheirCompositionality.pdf
	paper/abstract similarity: 0.9552369713783264

file: EnrichingWordVectorswithSubwordInformation.pdf
	paper/abstract similarity: 0.9469983577728271

file: LinguisticRegularitiesInContinuousSpaceWordRepresentations.pdf
	paper/abstract similarity: 0.9215396046638489

file: SiameseCBOW.pdf
	paper/abstract similarity: 0.9487900733947754



Comparing abstracts to the bodies of other papers led to some interesting results. Some abstracts were closest to their papers, but that was not the case for all of them. Without being too wordy, here are the results:

Files:
0 -> Distributed Representation of Words and Phrases and their Compositionality
1 -> Enriching Word Vectors with Subword Information
2 -> Linguistic Regularities in Continuous Space Word Representations
3 -> Siamese CBOW

* Abstract 0 ==> Closest to furthest paper embeddings: 0,3,1,2
* Abstract 1 ==> Closest to furthest paper embeddings: 0,3,1,2
* Abstract 2 ==> Closest to furthest paper embeddings: 0,3,1,2
* Abstract 3 ==> Closest to furthest paper embeddings: 0,3,1,2

In [25]:
# comparing abstract to other papers

print(f'Comparing the abstract of the following file to the bodies of other papers:\n\t{filenames[0]}\n')

for i in range(4):
    print(f'File to compare with: {filenames[i]}')
    print(f'\tSimilarity: {round(cosine_similarity(abstract_embeddings[3],body_embeddings[i]),4)}')



Comparing the abstract of the following file to the bodies of other papers:
	DistributedRepresentationsofWordsandPhrasesandtheirCompositionality.pdf

File to compare with: DistributedRepresentationsofWordsandPhrasesandtheirCompositionality.pdf
	Similarity: 0.9656999707221985
File to compare with: EnrichingWordVectorswithSubwordInformation.pdf
	Similarity: 0.9424999952316284
File to compare with: LinguisticRegularitiesInContinuousSpaceWordRepresentations.pdf
	Similarity: 0.9312000274658203
File to compare with: SiameseCBOW.pdf
	Similarity: 0.9488000273704529


# More Papers
After this, we add 5 papers randomly selected from the following subjects:
* Suspension Bridges
* North American Reptiles
* Chemical Engineering of Batteries
* TV's impact on cognitive development

The first thing we do is compare all paper's abstracts with its body again



In [38]:
num_files = 25

for i in range(25):
    print(f'File name: {filenames[i]}')
    print(f'\tAbstract/Paper Similarity: {compare_abstract_paper(i)}\n')

File name: HighEntropyElectrolytesforPracticalLithiumMetalBatteries.pdf
	Abstract/Paper Similarity: 0.9662533402442932

File name: ClimateFutureforLizards.pdf
	Abstract/Paper Similarity: 0.962161660194397

File name: DeterminationofReplacementCableForce.pdf
	Abstract/Paper Similarity: 0

File name: DistributedRepresentationsofWordsandPhrasesandtheirCompositionality.pdf
	Abstract/Paper Similarity: 0.9552369713783264

File name: incorporatingextraknowledgetoenhancewordembedding.pdf
	Abstract/Paper Similarity: 0.9703680276870728

File name: EnrichingWordVectorswithSubwordInformation.pdf
	Abstract/Paper Similarity: 0.9469983577728271

File name: lico2_iteration2.pdf
	Abstract/Paper Similarity: 0.979454755783081

File name: LinguisticRegularitiesInContinuousSpaceWordRepresentations.pdf
	Abstract/Paper Similarity: 0.9215396046638489

File name: ReptileResponsetoETLRoW.pdf
	Abstract/Paper Similarity: 0.9057645201683044

File name: DynamicBehaviorofSuspensionBridgesunderMovingLoads.pdf
	Abstra

We see quite a variance of similarities between abstracts and papers with this. I also discovered that a 2 papers were not preprocessed correctly, and because I don't particularly want to go searching again for papers on suspension bridges, the will just be omitted for now.

### Comparing across papers

First, we will sample an abstract from each group and compare it to the embeddings of all papers to find which are closest. 

In [54]:
# manually grouping vectors by content for comparison purposes
broken_embeddings = [2,9]
word_embedding_papers = [3,4,5,7,18]
suspension_bridge_papers = [16,20,23]
reptile_papers = [1,8,19,21,24]
battery_papers = [0,6,10,14,22]
tv_papers = [11,12,13,15,17]

def three_closest_papers(abstract_index: int):
    label_cosine_pairs = []
    for i in range(25):
        if i not in broken_embeddings:
            label_cosine_pairs.append((i,cosine_similarity(abstract_embeddings[abstract_index], body_embeddings[i])))
    label_cosine_pairs = sorted(label_cosine_pairs, key=lambda x: x[1], reverse=True)
    return label_cosine_pairs[:3]

def get_label_by_index(index: int) -> str:
    if index in word_embedding_papers:
        return 'Word Embeddings'
    if index in suspension_bridge_papers:
        return 'Suspension Bridges'
    if index in reptile_papers:
        return 'North American Reptiles'
    if index in battery_papers:
        return 'Battery Chemical Composition'
    if index in tv_papers:
        return 'TVs Cognitive Impact'

# grab paper from index 2 out of labled groups, then find closest 3 papers and report their topic.
output_groups = [[5],[23],[19],[10],[13]] # storing index of abstract compared and closest 3 paper bodies found to be printed later
for group in output_groups:
    group.append(three_closest_papers(group[0]))

    print(f'Abstract from {filenames[group[0]]}\n\tGroup: {get_label_by_index(group[0])}')
    print(f'\tClosest paper embeddings and their group')
    match_string = 'NOT ' if group[0] != int(group[1][0][0]) else ''
    print(f'\tAbstract DOES {match_string}belong to the closest paper')
    for i in range(3):
        print(f'\t\tSimilarity: {round(float(group[1][i][1]),4)} Group: {get_label_by_index(group[1][i][0])}')
    print()    

Abstract from EnrichingWordVectorswithSubwordInformation.pdf
	Group: Word Embeddings
	Closest paper embeddings and their group
	Abstract DOES NOT belong to the closest paper
		Similarity: 0.9728 Group: TVs Cognitive Impact
		Similarity: 0.9724 Group: Battery Chemical Composition
		Similarity: 0.972 Group: TVs Cognitive Impact

Abstract from FlutterAnalysisofSuspensionBridges.pdf
	Group: Suspension Bridges
	Closest paper embeddings and their group
	Abstract DOES NOT belong to the closest paper
		Similarity: 0.9648 Group: Battery Chemical Composition
		Similarity: 0.9574 Group: Suspension Bridges
		Similarity: 0.9536 Group: North American Reptiles

Abstract from AmphibiansandReptilesofUSDOD.pdf
	Group: North American Reptiles
	Closest paper embeddings and their group
	Abstract DOES NOT belong to the closest paper
		Similarity: 0.8376 Group: Suspension Bridges
		Similarity: 0.8345 Group: North American Reptiles
		Similarity: 0.8139 Group: TVs Cognitive Impact

Abstract from ChallengesInSo

The results are interesting. Only the abstracts about batteries and TV's impacts get at least 2 of the top 3 of the closest papers to be in the same group, and the `ChallengesInSolidStateBatteries.pdf`'s abstract actually pairs with its respective body's embedding.

### Final Thoughts

I planned to run one final experiment that would use a clustering algorithm to see if the paper embeddings were near each other, but due to the small number of papers per group, I decided not to carry through with it since there likely won't be enough data to pull any sort of usefull meaning from.

Again I've found myself thinking that more data would have made this project more insightful. 

Things I can take away:
* I initially thought that the abstracts' embeddings would be very close to their paper's embeddings; however, that was not the case. I learned that an in-depth text does not have the same embedding as a summary of the content; although, they do tend to have a nearness
* Document embeddings may be useful in differentiating between very broad ideas, but as the scope narrows, they don't indicate much difference. Writing styles may also heavily impact measurements in this manner.
* Preprocessing can be quite the endeavor, but LLMs are a great tool to get you kickstarted, even without much experience.